# **???? 3.1: ?????????? ???????: Logistic Regression**

# **??? 1: ?????????? ????????????**

**1.1 ??????????? ?????? ? ???????**

In [ ]:
import io
from sklearn.pipeline import Pipeline
import pandas as pd
import dill
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.metrics import classification_report, plot_confusion_matrix
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
def get_classification_report(y_train_true, y_train_pred, y_test_true, y_test_pred):
    print('TRAIN\n\n' + classification_report(y_train_true, y_train_pred))
    print('TEST\n\n' + classification_report(y_test_true, y_test_pred))


**1.2 ???????? ????????? ??????????????**


In [ ]:
with open('../EDA_and_Preprocessing/prep_with_cat.dill', 'rb') as prep_cat_pipeline:
    prep_cat = dill.load(prep_cat_pipeline)

**1.3 ???????? ??????**

In [ ]:
TRAIN_DATA_PATH = '../data/X_train.csv'
TRAIN_TARGET_PATH = '../data/y_train.csv'

TEST_DATA_PATH = '../data/X_test.csv'
TEST_TARGET_PATH = '../data/y_test.csv'

In [ ]:
train_base = pd.read_csv(TRAIN_DATA_PATH)
train = train_base.copy()
train_target_base = pd.read_csv(TRAIN_TARGET_PATH)
train_target = train_target_base.copy()
display(train.shape, train_target.shape)

In [ ]:
test_base = pd.read_csv(TEST_DATA_PATH)
test = test_base.copy()
test_target_base = pd.read_csv(TEST_TARGET_PATH)
test_target = test_target_base.copy()
display(test.shape, test_target.shape)

# **??? 2: ?????????? ??????**

**2.1 Baseline**

?????? ?????? ? ??????????? ?? ????????? ?? ????????????? ???????.

In [ ]:
classifier = Pipeline([
    ('preprocessing', prep_cat),
    ('classifier', LogisticRegression(random_state = 0)),
])

classifier.fit(train, train_target)

In [ ]:
train_pred_proba = classifier.predict_proba(train)
train_pred = classifier.predict(train)
test_pred_proba = classifier.predict_proba(test)
test_pred = classifier.predict(test)
print(f'Train ROC_AUC: {roc_auc_score(train_target, train_pred_proba.T[1])}')
print(f'Train F1 SCORE: {f1_score(train_target, train_pred)}')
print(f'Test ROC_AUC: {roc_auc_score(test_target, test_pred_proba.T[1])}')
print(f'Test F1 SCORE: {f1_score(test_target, test_pred)}')

In [ ]:
get_classification_report(train_target['HeartDisease'], train_pred, test_target['HeartDisease'], test_pred)

In [ ]:
plot_confusion_matrix(classifier, test, test_target['HeartDisease'], cmap=plt.cm.Blues);

**2.2 ?????? ???????????????**

?????????????? ????????? ? ??????? GridSearchCV ? ??????????????? ?? 5 ??????.

In [ ]:
%%time
cv = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

frozen_params = {
    'random_state': 0
}
parameters = {
    'C': np.logspace(-4, 4, 20), 
    'class_weight': [None, 'balanced'], 
    'penalty': ['l1', 'l2'],
    'solver': ['lbfgs', 'liblinear'],
}

clf = GridSearchCV(LogisticRegression(**frozen_params), parameters, n_jobs=-1, verbose=3, scoring='f1' ,cv=cv)

classifier = Pipeline([
    ('preprocessing', prep_cat),
    ('classifier', clf),
])
classifier.fit(train, train_target.values.ravel())

**2.3 ???????? ??????**

?????? ?????? ? ??????? ??????????? ?? ????????????? ??????? ? ????????? ???????? ???????? ?? ???????? ???????.

In [ ]:
print(clf.best_score_)
print(clf.best_params_)

In [ ]:
classifier = Pipeline([
    ('preprocessing', prep_cat),
    ('classifier', LogisticRegression(**clf.best_params_, random_state = 0)),
])
classifier.fit(train, train_target)

In [ ]:
train_pred_proba = classifier.predict_proba(train)
train_pred = classifier.predict(train)
test_pred_proba = classifier.predict_proba(test)
test_pred = classifier.predict(test)

In [ ]:
get_classification_report(train_target['HeartDisease'], train_pred, test_target['HeartDisease'], test_pred)

In [ ]:
plot_confusion_matrix(classifier, test, test_target['HeartDisease'], cmap=plt.cm.Blues)


# **??? 3: ?????????? ????????? ??????**

In [ ]:
with open("model_LogisticRegressionClassifier.dill", "wb") as f:
    dill.dump(classifier, f)